# 02 - Ablation Study

Mục tiêu:
- Ablation thành phần: tắt Split, tắt Merge, tắt cả hai.
- Ablation siêu tham số λ: kiểm tra độ nhạy với λ ∈ {1.5, 2.0, 2.5, 3.0}.


In [ ]:
import sys
import pandas as pd
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

sys.path.append('..')
from src.utils import set_seed, get_device, extract_features, train_clustering

set_seed(42)
device = get_device()
print(f'[INFO] device = {device}')


In [ ]:
# Two-step feature extraction trên CIFAR-10
resnet = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
backbone = nn.Sequential(*list(resnet.children())[:-1]).to(device)

transform_cifar = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

cifar_test = torchvision.datasets.CIFAR10(root='../data', train=False, download=True, transform=transform_cifar)
subset = torch.utils.data.Subset(cifar_test, range(3000))
loader = DataLoader(subset, batch_size=128, shuffle=False)

X, y = extract_features(loader, backbone, device)
print('[INFO] features:', X.shape, '| labels:', y.shape)


In [ ]:
def run_cfg(name, k0=30, enable_split=True, enable_merge=True, lambda_param=2.0, epochs=30):
    k_star, acc, nmi, ari = train_clustering(
        features=X, labels=y, k_max=k0, device=device,
        epochs=epochs, lr=1e-3,
        lambda_param=lambda_param,
        enable_split=enable_split, enable_merge=enable_merge
    )
    return {
        'Setting': name,
        'K0': k0,
        'lambda': lambda_param,
        'Split': enable_split,
        'Merge': enable_merge,
        'K*': int(k_star),
        'ACC(%)': acc*100,
        'NMI(%)': nmi*100,
        'ARI(%)': ari*100
    }


In [ ]:
# Ablation 1: thành phần Split/Merge
component_rows = [
    run_cfg('Full PnP', enable_split=True, enable_merge=True),
    run_cfg('No Split', enable_split=False, enable_merge=True),
    run_cfg('No Merge', enable_split=True, enable_merge=False),
    run_cfg('No Split + No Merge (Fixed-K)', enable_split=False, enable_merge=False),
]
comp_df = pd.DataFrame(component_rows)
comp_df


In [ ]:
# Ablation 2: quét lambda
lambda_rows = []
for lam in [1.5, 2.0, 2.5, 3.0]:
    lambda_rows.append(run_cfg(f'lambda={lam}', lambda_param=lam, enable_split=True, enable_merge=True))

lambda_df = pd.DataFrame(lambda_rows)
lambda_df


## Nhận xét ablation (điền vào báo cáo)

- Nếu tắt Split hoặc Merge mà hiệu năng giảm, điều đó xác nhận từng module đều đóng góp.
- Nếu K* thay đổi mạnh khi đổi λ, có thể kết luận mô hình nhạy với λ trên dữ liệu thực nghiệm của nhóm.
- Đây là bằng chứng cho nhận định phê bình: bài toán chọn K được chuyển thành bài toán chọn λ.
